# Spatial Autocorrelation Diagnostics
Evaluating Moran's I and LISA to prove spatial clustering.

In [1]:
import pandas as pd
import geopandas as gpd
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local
import matplotlib.pyplot as plt

df = pd.read_csv('../data/processed/spatial_panel.csv')
gdf = gpd.read_file('../data/raw/geospatial/prefectures.geojson')

# Clean GeoJSON names to match prefecture_en
def clean_geo_name(name):
    name = str(name).replace(" To", "").replace(" Fu", "").replace(" Ken", "")
    name = name.replace("Hokkai Do", "Hokkaido")
    return name.strip()
    
gdf["nam"] = gdf["nam"].apply(clean_geo_name)
gdf = gdf.rename(columns={"nam": "prefecture_en"})

merged = gdf[["prefecture_en", "geometry"]].merge(df, on="prefecture_en", how="inner")
merged = merged.to_crs("EPSG:3857")

w = KNN.from_dataframe(merged, k=4)
w.transform = 'r'

In [2]:
# Global Moran's I
y = merged['target_pop_change_pct'].values
moran = Moran(y, w)
print(f"Global Moran's I: {moran.I:.4f}")
print(f"p-value: {moran.p_sim:.4f}")

if moran.p_sim < 0.05:
    print("Significant spatial autocorrelation detected!")
else:
    print("No significant spatial autocorrelation.")

Global Moran's I: 0.4430
p-value: 0.0010
Significant spatial autocorrelation detected!
